# Train-only preprocessing and CPU feasibility pilot
Run cells in order. Notebook outputs remain visible; durable artifacts are written to results/v2.


In [1]:
from pathlib import Path
import os, sys, json, errno, time
# VS Code + Colab: load the synchronized source folder, without browser uploads.
DRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None
def with_drive_retry(operation, attempts=3, reconnect=None):
    for attempt in range(attempts):
        try:
            return operation()
        except OSError as exc:
            disconnected = exc.errno in {errno.ENOTCONN, errno.EIO, errno.ESTALE} or "Transport endpoint" in str(exc)
            if not disconnected or (drive is None and reconnect is None) or attempt == attempts-1:
                raise
            print(f"Drive disconnected. Reconnecting and retrying ({attempt+1}/{attempts-1})...", flush=True)
            if reconnect is not None:
                reconnect()
            else:
                drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def ensure_drive():
    if drive is not None:
        if not DRIVE.is_dir():
            drive.mount("/content/drive", force_remount=False)
        # Probe the filesystem: an existing mount directory can still be disconnected.
        next(DRIVE.iterdir(), None)
with_drive_retry(ensure_drive)

def resolve_v2(bases):
    checked = []
    for base in bases:
        for candidate in (base, base/"implementation/v2", base/"v2"):
            checked.append(str(candidate))
            required = ["src/atlas_v2/__init__.py", "src/atlas_v2/data.py",
                        "src/atlas_v2/models.py", "src/atlas_v2/experiments.py", "requirements.txt"]
            if all((candidate/name).is_file() for name in required):
                return candidate.resolve()
    raise FileNotFoundError(
        "The v2 Python source is not visible to this kernel. "
        "Expected the synchronized implementation/v2 folder on mounted Drive. "
        "Set ATLAS_V2_ROOT to that folder if your Drive location differs. Checked: "
        + "; ".join(checked))

manual = os.environ.get("ATLAS_V2_ROOT")
bases = [Path(manual).expanduser()] if manual else []
bases += [DRIVE/"projects/implementation/ATLAS-IDNet/implementation/v2",
          DRIVE/"projects/ATLAS-IDNet", DRIVE/"ATLAS-IDNet",
          Path.cwd(), *Path.cwd().parents, Path("/content/ATLAS-IDNet")]
V2 = with_drive_retry(lambda: resolve_v2(bases))
sys.path.insert(0, str(V2/"src"))
for name in list(sys.modules):
    if name == "atlas_v2" or name.startswith("atlas_v2."):
        del sys.modules[name]
import atlas_v2
assert Path(atlas_v2.__file__).resolve().parent == (V2/"src/atlas_v2").resolve()
print("Code:", V2)
print("Source import OK. No file upload or model weights required.")


Code: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/implementation/v2
Source import OK. No file upload or model weights required.


In [2]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(V2/"requirements.txt")])
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")
import torch, pandas, sklearn
print("CPU runtime:", torch.__version__, "threads:", os.cpu_count())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU runtime: 2.11.0+cpu threads: 2


In [3]:
# Set this explicitly if your results directory differs.
default = "/content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/experiment_config.json"
CONFIG_PATH = Path(os.environ.get("ATLAS_V2_CONFIG", default if Path("/content").exists() else str(V2/"results/experiment_config.json")))
CONFIG = json.loads(CONFIG_PATH.read_text())
RESULTS = Path(CONFIG["results"])


In [4]:
from atlas_v2.data import prepare, PREPARATION_VERSION
import hashlib
for name, spec in CONFIG["datasets"].items():
    spec = {k: v for k, v in spec.items() if k != "prepared"}
    key = hashlib.sha256(json.dumps({"spec": spec, "preparation_version": PREPARATION_VERSION}, sort_keys=True).encode()).hexdigest()[:12]
    folder = RESULTS/name/("prepared_"+key)
    # Existing prepared data is immutable. Change the spec to create a distinct preparation.
    def prepare_one():
        if not (folder/"metadata.json").exists():
            prepare(spec, folder)
        print(name, json.loads((folder/"metadata.json").read_text())["counts"])
        CONFIG["datasets"][name]["prepared"] = str(folder)
        temp = CONFIG_PATH.with_suffix(".tmp")
        temp.write_text(json.dumps(CONFIG, indent=2))
        os.replace(temp, CONFIG_PATH)
    with_drive_retry(prepare_one)


File 1/169: checksumming /content/drive/MyDrive/projects/Dataset/Security/CICIoT2023/csv/CICIoT2023/part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Sampling part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
File 2/169: checksumming /content/drive/MyDrive/projects/Dataset/Security/CICIoT2023/csv/CICIoT2023/part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Sampling part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
File 3/169: checksumming /content/drive/MyDrive/projects/Dataset/Security/CICIoT2023/csv/CICIoT2023/part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Sampling part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
File 4/169: checksumming /content/drive/MyDrive/projects/Dataset/Security/CICIoT2023/csv/CICIoT2023/part-00003-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Sampling part-00003-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
File 5/169: checksumming /content/drive/MyDrive/projects/Dataset/Security/CICIoT2023/csv/CICIoT2023/part-000

In [5]:
from atlas_v2.experiments import run
for name, spec in CONFIG["datasets"].items():
    path = run(spec["prepared"], RESULTS/name/"pilot", epochs=2, seed=42)
    print(name, json.loads((path/"metrics.json").read_text()))
print("Pilot only: not final paper results. Review duration before notebook 02.")


atlas_ce_42_7516f446400a8710 {'epoch': 1, 'loss': 0.832809506296105, 'val_macro_f1': 0.39168668622742947}
atlas_ce_42_7516f446400a8710 {'epoch': 2, 'loss': 0.48045891098337584, 'val_macro_f1': 0.4310488741509722}
CICIoT2023 {'accuracy': 0.818977153426986, 'macro_f1': 0.4429633956584446, 'weighted_f1': 0.7903675149367074, 'mcc': 0.8043128408494825, 'invocation_seconds': 24.32096321500103}
atlas_ce_42_966f33223210a116 {'epoch': 1, 'loss': 0.6251923963236686, 'val_macro_f1': 0.3229044580192114}
atlas_ce_42_966f33223210a116 {'epoch': 2, 'loss': 0.29111606848574784, 'val_macro_f1': 0.41999904925684506}
NF-UQ-NIDS-v2 {'accuracy': 0.9338842975206612, 'macro_f1': 0.40646876473100957, 'weighted_f1': 0.9323374233525858, 'mcc': 0.911576723039089, 'invocation_seconds': 17.76096909900116}
Pilot only: not final paper results. Review duration before notebook 02.
